In [18]:
!pip install chromadb sentence-transformers openpyxl --quiet
!pip install groq --quiet

In [22]:
import os
import json
import pandas as pd
import gc
import chromadb
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import warnings
from transformers import logging

In [23]:
warnings.filterwarnings("ignore")
logging.set_verbosity_error()
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [24]:
file_name = '/kaggle/input/datasets/samvelgalstyan/dataset/24.08.2026 11.02.51.xlsx'

if not os.path.exists(file_name):
    raise FileNotFoundError(f"Файл по пути {file_name} не найден!")

df = pd.read_excel(file_name)

columns_map = {str(col).strip().lower(): col for col in df.columns}
name_col = columns_map.get('название', df.columns[0])
art_col = columns_map.get('артикул', df.columns[1] if len(df.columns) > 1 else df.columns[0])
group_col = columns_map.get('группа', df.columns[2] if len(df.columns) > 2 else df.columns[0])

df = df.rename(columns={name_col: 'Название', art_col: 'Артикул', group_col: 'Группа'})
df['Название'] = df['Название'].astype(str).fillna('').str.strip()
df['Артикул'] = df['Артикул'].astype(str).fillna('').str.strip()
df['Группа'] = df['Группа'].astype(str).fillna('').str.strip()

chroma_client = chromadb.PersistentClient(path="./real_iiko_db")
collection = chroma_client.get_or_create_collection(name="yerevan_nomenclature")
embedding_model = SentenceTransformer("sentence-transformers/LabSE")

def get_vector(text):
    return embedding_model.encode(text).tolist()

if collection.count() == 0:
    print(f"Импортируем {len(df)} строк пачками...")
    names = df['Название'].tolist()
    groups = df['Группа'].tolist()
    articles = df['Артикул'].tolist()
    
    all_embeddings = embedding_model.encode(names, batch_size=256, show_progress_bar=True).tolist()
    
    batch_size = 500
    for i in range(0, len(names), batch_size):
        end_idx = min(i + batch_size, len(names))
        metadata_batch = [{"group": groups[j], "article": articles[j]} for j in range(i, end_idx)]
        id_batch = [f"{art}_{j}" if art != 'nan' and art != '' else f"auto_{j}" for j, art in enumerate(articles[i:end_idx], start=i)]
        
        collection.upsert(embeddings=all_embeddings[i:end_idx], documents=names[i:end_idx], metadatas=metadata_batch, ids=id_batch)
    print("🚀 Векторный индекс успешно создан!")

print("Загрузка локальной языковой модели Qwen 2.5...")
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
llm_pipeline = pipeline("text-generation", model=llm_model, tokenizer=tokenizer)

def extract_specs_locally_with_llm(product_name, predicted_group):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    prompt = f"""
    Ты — ИИ-агент автоматизации ресторанного учета iiko. Твоя задача — проанализировать сырое название товара и строго извлечь его характеристики БЕЗ смешивания с предыдущими запросами.
    
    Входные данные:
    - Название товара от поставщика: "{product_name}"
    - Группа хранения в iiko: "{predicted_group}"
    
    Сформируй чистый JSON-ответ (без markdown, без ```json, без лишнего текста) строго по следующей схеме:
    {{
        "clean_name": "Короткое очищенное название. Международные бренды (Coca-Cola, Fanta, Sprite) пиши строго на английском латиницей. Локальные/русские товары (Мыло, Молоко, Сметана) пиши на русском языке.",
        "brand": "Точное название бренда товара. На английском для мировых брендов (Coca-Cola) и на русском для локальных/российских брендов (Простоквашино). Если бренд не указан и товар локальный общего типа (например, мыло жидкое), пиши 'Локальный бренд'",
        "volume_or_weight": Числовое значение объема или веса одной единицы товара в виде числа (например: 0.5, 5.0, 1.0). Если данных нет, пиши null,
        "unit": "Единица измерения, строго одна из: л, мл, кг, гр, шт",
        "fat_percentage": Процент жирности для молочных продуктов в виде числа (например: 3.2). Для остальных товаров, включая мыло и колу, пиши 'Не применимо'
    }}
    """
    
    messages = [{"role": "user", "content": prompt}]
    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    try:
        outputs = llm_pipeline(text_prompt, max_new_tokens=256, do_sample=False, temperature=0.0)
        ai_response = outputs[0]["generated_text"][len(text_prompt):].strip()
        if "```" in ai_response:
            ai_response = ai_response.split("```")[1]
            if ai_response.startswith("json"):
                ai_response = ai_response[4:]
                
        return json.loads(ai_response.strip())
    except Exception as e:
        return {"Ошибка ИИ-парсинга": str(e)}

def smart_ai_agent(new_product_name):
    query_vector = get_vector(new_product_name)
    results = collection.query(query_embeddings=[query_vector], n_results=1)
    
    try: best_match_name = results['documents'][0][0]
    except: best_match_name = "Не найдено"
        
    try: predicted_group = results['metadatas'][0][0]['group']
    except: predicted_group = "Не определена"
        
    try: distance = results['distances'][0][0]
    except: distance = 1.0
    
    llm_features = extract_specs_locally_with_llm(new_product_name, predicted_group)

    decision = {
        "Статус": "Успешно обработано локальным LLM-агентом iiko",
        "Куда добавить (Определенная Группа)": predicted_group,
        "Конкретные характеристики товара (Локальный ИИ)": llm_features,
        "Ближайший аналог в вашем Excel": best_match_name,
        "Уверенность (ближе к 0 — точнее)": round(distance, 4)
    }
    
    return json.dumps(decision, ensure_ascii=False, indent=4)

# ==========================================
#  ЗАПУСК ТЕСТОВ
# ==========================================
print("\n=== ЗАПУСК ТЕСТИРОВАНИЯ ЛОКАЛЬНОГО LLM-АГЕНТА ===")
print(smart_ai_agent("Coca-Cola vanilla 0.5л"))
print(smart_ai_agent("Молоко ультрапаст Простоквашино 3.2% 1л"))
print(smart_ai_agent("Жидкое мыло для рук 5 литров"))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Загрузка локальной языковой модели Qwen 2.5...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


=== ЗАПУСК ТЕСТИРОВАНИЯ ЛОКАЛЬНОГО LLM-АГЕНТА ===
{
    "Статус": "Успешно обработано локальным LLM-агентом iiko",
    "Куда добавить (Определенная Группа)": "Ջրեղեն",
    "Конкретные характеристики товара (Локальный ИИ)": {
        "clean_name": "Coca-Cola vanilla 0.5л",
        "brand": "Coca-Cola",
        "volume_or_weight": 0.5,
        "unit": "л",
        "fat_percentage": "Не применимо"
    },
    "Ближайший аналог в вашем Excel": "Напиток освежающий Coca-cola vanilla 0,35л ж/б",
    "Уверенность (ближе к 0 — точнее)": 0.202
}
{
    "Статус": "Успешно обработано локальным LLM-агентом iiko",
    "Куда добавить (Определенная Группа)": "Կաթնամթերք",
    "Конкретные характеристики товара (Локальный ИИ)": {
        "clean_name": "молоко",
        "brand": "Простоквашино",
        "volume_or_weight": 1.0,
        "unit": "л",
        "fat_percentage": 3.2
    },
    "Ближайший аналог в вашем Excel": "Կաթ մարիլա 3.2% 1լ",
    "Уверенность (ближе к 0 — точнее)": 0.6607
}
{
    "Статус